# Greeks and Greek Uncertainty

Greeks measure local option risk sensitivities. This notebook calculates Greeks using mid implied volatility and then measures how those Greeks change when bid and ask implied volatilities are used instead.

## Greek definitions

Delta measures sensitivity to the underlying stock price. Gamma measures how quickly Delta changes. Vega measures sensitivity to volatility. Theta measures time decay. Rho measures sensitivity to the risk-free rate.

Greeks are local approximations, so they describe small changes around the current input values. They are not guarantees for large market moves.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

In [ ]:
project_root = Path.cwd()

if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

In [ ]:
from src.greek_uncertainty import (
    greek_failure_summary,
    greek_uncertainty_summary,
    insert_greek_results_to_database,
    plot_all_greek_uncertainty_heatmaps,
    prepare_greek_uncertainty_dataset,
)

## Load IV uncertainty data

The input dataset should contain `IV_bid`, `IV_mid`, and `IV_ask` from the implied-volatility workflow.

In [ ]:
input_path = project_root / "data" / "processed" / "iv_uncertainty_analysis.csv"

if not input_path.exists():
    input_path = project_root / "data" / "processed" / "iv_uncertainty_results.csv"

if not input_path.exists():
    raise FileNotFoundError("No IV uncertainty dataset found.")

iv_quotes = pd.read_csv(input_path)
iv_quotes.head(10)

## Calculate Greeks from bid, mid, and ask IV

The mid-IV Greeks are used as the central estimate. The bid-IV and ask-IV Greeks show how quote uncertainty changes the risk sensitivities.

In [ ]:
risk_free_rate = 0.04

greek_results, greek_wide = prepare_greek_uncertainty_dataset(
    iv_quotes,
    risk_free_rate=risk_free_rate,
    retained_only=True,
)

greek_results.head(10)

In [ ]:
display_columns = [
    column
    for column in [
        "contractSymbol",
        "expiration",
        "option_type",
        "strike",
        "Delta_mid",
        "Gamma_mid",
        "Vega_mid",
        "Theta_mid",
        "Delta_range",
        "Gamma_range",
        "Vega_range",
        "Theta_range",
    ]
    if column in greek_wide.columns
]

greek_wide[display_columns].head(15)

## Save Greek outputs

In [ ]:
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(exist_ok=True)

greek_results_path = processed_dir / "greek_results.csv"
greek_uncertainty_path = processed_dir / "greek_uncertainty_results.csv"

greek_results.to_csv(greek_results_path, index=False)
greek_wide.to_csv(greek_uncertainty_path, index=False)

greek_results_path, greek_uncertainty_path

## Greek uncertainty summary

In [ ]:
summary = greek_uncertainty_summary(greek_wide)
summary

In [ ]:
summary_path = processed_dir / "greek_uncertainty_summary.csv"
summary.to_csv(summary_path, index=False)

summary_path

## Greek calculation failures

In [ ]:
failure_summary = greek_failure_summary(greek_results)
failure_summary

## Heatmaps

In [ ]:
figures_dir = project_root / "figures"
figures_dir.mkdir(exist_ok=True)

plot_all_greek_uncertainty_heatmaps(
    greek_wide,
    output_dir=figures_dir,
)

## Optional database insert

This cell inserts Greek outputs into the existing SQLite database when the database and matching cleaned quote identifiers are available.

In [ ]:
db_path = project_root / "data" / "database" / "options_frictions_lab.db"

if db_path.exists() and "snapshot_id" in iv_quotes.columns:
    snapshot_id = int(iv_quotes["snapshot_id"].dropna().iloc[0])
    database_insert_summary = insert_greek_results_to_database(
        db_path=db_path,
        greek_results=greek_results,
        greek_wide=greek_wide,
        snapshot_id=snapshot_id,
    )
else:
    database_insert_summary = {
        "status": "skipped",
        "reason": "database file or snapshot_id not available",
    }

database_insert_summary

## Interpretation

Greek uncertainty comes from IV uncertainty. If a contract has a wide bid-ask spread, the bid-IV and ask-IV can be far apart. Delta, Gamma, Vega, and Theta calculated from those IV values can also be far apart, which makes the option's risk measurement less precise.